In [0]:
# Bronze ingestion: ALL nginx/Apache Combined Log Format files from
# /Volumes/b2b_ecommerce/raw/logs -> bronze.logs streaming table.
# Auto Loader reads each .log as text, one row per line: the line is stored EXACTLY as written
# in the file - no parsing, no cleaning, no filtering - so malformed lines (e.g. '-- GARBAGE --')
# are ingested untouched like any other line; parsing into columns belongs to the silver layer.
# Tracking columns are added alongside, not instead: source_file (the file the line came from,
# same convention as bronze.leads) and ingested_at / updated_at (both set at insert time - the
# table is append-only, so a row is never updated afterwards).
# source_file comes from Auto Loader's built-in _metadata.file_path (documented pattern for
# tracing rows back to source files; also joinable against cloud_files_state() for the file
# lifecycle). The stream stays pure Spark - no per-file Python - so it scales to many/large
# files. ingested_at uses current_timestamp(), which is evaluated per processed micro-batch.
# pathGlobFilter "*.log" is a file-source option: only files whose names match the glob are
# picked up; anything else that lands in the folder (e.g. _SUCCESS markers, temp/part files)
# is ignored by the stream.
# Failure behaviour: each definition step is logged and wrapped so a failure names the failed
# step; the pipeline event log keeps the WARN trail of steps and any ERROR that aborts the flow.
import logging

from pyspark import pipelines as dp
from pyspark.sql import functions as F

logger = logging.getLogger("bronze.logs")

SOURCE_PATH = "/Volumes/b2b_ecommerce/raw/logs"


def _step(step: str, action):
    """Log a definition step at WARN (lands in the pipeline event log) and fail with context."""
    logger.warning("logs: %s", step)
    try:
        return action()
    except Exception as e:
        logger.error("logs: %s FAILED - %s: %s", step, type(e).__name__, e)
        raise ValueError(f"logs: {step} failed - {type(e).__name__}: {e}") from None


@dp.table(
    name="bronze.logs",
    comment="Raw nginx/Apache combined-format access log lines from /Volumes/b2b_ecommerce/raw/logs, kept exactly as written (one row per line) with source_file, ingested_at and updated_at tracking columns.",
)
def logs():
    stream = _step(
        "open Auto Loader text stream on /Volumes/b2b_ecommerce/raw/logs",
        lambda: (
            spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "text")
            .option("pathGlobFilter", "*.log")
            .load(SOURCE_PATH)
        ),
    )
    return _step(
        "publish raw log lines with tracking columns",
        lambda: stream.select(
            F.col("value").alias("log_line"),
            F.col("_metadata.file_path").alias("source_file"),
            F.current_timestamp().alias("ingested_at"),
            F.current_timestamp().alias("updated_at"),
        ),
    )